In [ ]:
'''imports'''
from string import digits
from skimage import io
from skimage.color import rgba2rgb
from torch.autograd import Variable

import cv2
import pandas
import os
import shutil
import matplotlib.pyplot as plt
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from PIL import Image, ImageOps

In [ ]:
path = '/kaggle/input/best-artworks-of-all-time/resized/resized/'
#get number of files in the directory
number_of_input_files = len(os.listdir(path))
print(number_of_input_files)

In [ ]:
'''create labels -> and correction of spelling mistakes'''

labels = []

for i in range(number_of_input_files):
    label = os.listdir(path)[i].translate(str.maketrans('', '', digits))
    label = label.translate(str.maketrans('', '', '_'))
    label = label.translate(str.maketrans('', '', '.jpg'))
    label = label.translate(str.maketrans('', '', '-'))
    if label.find('Albrecht') != -1:
        label = 'AlbrechtDuerer'
    if label.find('AmedeoModiliani') != -1:
        label = 'AmedeoModigliani'
    if label.find('VincentvanGoh') != -1:
        label = 'VincentvanGogh'
    if label.find('PaulGauuin') != -1:
        label = 'PaulGauguin'
    if label.find('Rahael') != -1:
        label = 'Raphael'
    if label.find('Michelanelo') != -1:
        label = 'Michelangelo'
    if label.find('DieoVelazquez') != -1:
        label = 'DiegoVelazquez'
    if label.find('DieoRivera') != -1:
        label = 'DiegoRivera'
    if label.find('ReneMaritte') != -1:
        label = 'ReneMagritte'
    if label.find('EueneDelacroix') != -1:
        label = 'EugeneDelacroix'
    if label.find('MarcChaall') != -1:
        label = 'MarcChagall'
    if label.find('PierreAuusteRenoir') != -1:
        label = 'PierreAugusteRenoir'
    if label.find('Caravaio') != -1:
        label = 'Caravaggio'
    if label.find('EdarDeas') != -1:
        label = 'EdgarDegas'
    if label.find('PieterBrueel') != -1:
        label = 'PieterBruegel'
    if label.find('GeoresSeurat') != -1:
        label = 'GeorgesSeurat'        
    labels.append(label)

#print(labels)

In [ ]:
#print(labels)
count = labels.count('GeorgesSeurat')
print(count)

In [ ]:
'''define classes'''

# the following code was used to verify Spelling mistake in the data 

path = '/kaggle/input/best-artworks-of-all-time/artists.csv'

#get number of filed in the directory
csv = pandas.read_csv(path)
name = csv['name']

classes1 = ()

for i in name:
    i = i.translate(str.maketrans('', '', '-'))
    i = i.translate(str.maketrans('', '', ' '))
    classes1 = classes1 + (i,)

print(classes1)


# OR

classes2 = ()

for i in labels:
    if i not in classes2:
        classes2 = classes2 + (i,)

print(classes2)

#--------------------------------------------
#AlbrechtDürer was renamed AlbrechtDuerer to avoid later complications 
print('--------')
for i in classes1:
    if i not in classes2:
        print(i)
        
print('--------')
for i in classes2:
    if i not in classes1:
        print(i) 

In [ ]:
'''
reshape size of images
version 1: use smalles image and cut the others
version 2: use a black/white framework for all other images
'''
path = '/kaggle/input/best-artworks-of-all-time/resized/resized/'

big_m = 99999

smallest_width = big_m
biggest_width = 0
smallest_height = big_m
biggest_height = 0

for i in range(number_of_input_files):
    img_path = os.listdir(path)[i]
    #print(img_path)
    img = io.imread(path + img_path)
    #print(img.type) #prints numpy.ndarray
    #print(img.shape) #-> shows differnt sizes of the images
    
    if img.shape[0] >= biggest_height:
        biggest_height = img.shape[0]
    elif img.shape[0] <= smallest_height:
        smallest_height = img.shape[0]
    
    if img.shape[1] >= biggest_width:
        biggest_width = img.shape[1]
    elif img.shape[1] <= smallest_width:
        smallest_width = img.shape[1]
         
print(smallest_width, biggest_width, smallest_height, biggest_height)

In [ ]:
#128x128 pixel because of beneficial properties 
width = 1000
height = 1000

#problem: 
small_image = cv2.resize(img, dsize=(width, height), interpolation=cv2.INTER_CUBIC)
plt.figure()
plt.imshow(img)

plt.figure()
plt.imshow(small_image)


In [ ]:
'''problem: images have different dimensions'''

path = '/kaggle/input/best-artworks-of-all-time/resized/resized/'

img_path = os.listdir(path)[21]
img = io.imread(path + img_path)
print('dimension:', img.shape)
plt.figure()
plt.imshow(img)

img_path = os.listdir(path)[22]
img = io.imread(path + img_path)
print('dimension:', img.shape)
plt.figure()
plt.imshow(img)

img_path = os.listdir(path)[22]
img = io.imread(path + img_path)
print('dimension:', img.shape)
plt.figure()
plt.imshow(img[:, :, 0])

print(img.shape[0])
#result: -> all images are saved in/transformed to rgb to not loose information 

In [ ]:
# create new folder for the adjusted images 
working_folder = '../working/adapted_images/'

# if there is an 'old' folder -> delete it 
if os.path.exists(working_folder):shutil.rmtree(working_folder)
# create a new one 
if not os.path.exists(working_folder):os.makedirs(working_folder)

In [ ]:
'''edit, save, define data'''
path = '/kaggle/input/best-artworks-of-all-time/resized/resized/'

classes3 = []

for i in labels:
    original_number_images = labels.count(i)
    if original_number_images >= 80 and i not in classes3:
        classes3.append(i)

for i in classes3: 
    new_folder = '../working/adapted_images/' + i + '/'
    if not os.path.exists(new_folder):os.makedirs(new_folder)
        
print(classes3)

In [ ]:
for i in range(number_of_input_files):
    label = labels[i]
    original_number_images = labels.count(labels[i])
    if original_number_images >= 80:
        number_of_images = len(os.listdir(working_folder + '/' + label))
        if number_of_images <= 80:
            img_path = os.listdir(path)[i]
            #print(img_path)
            img = io.imread(path + img_path)
            #resize image to defined width and height
            old_size = (img.shape[0], img.shape[1])

            img = Image.open(path + img_path)
            new_im = Image.new("RGB", (height, width))
            if old_size[0] > width or old_size[1] > height:
                img = img.resize((width, height))
            new_im.paste(img, (int((height-old_size[1])/2), int((width-old_size[0])/2)))


            #plt.figure()
            #plt.imshow(new_im)

            '''
            try:
                new_size = (width, height)
                new_im = Image.new("RGB", new_size)   ## luckily, this is already black!
                new_im.paste(img, ((new_size[0]-old_size[0])//2,(new_size[1]-old_size[1])//2))
                #img = ImageOps.expand(Image.open(img),border=300,fill='black')
                #small_image = cv2.resize(img[:, :, :], dsize=(width, height), interpolation=cv2.INTER_CUBIC)
            except:
                # if image is two dimensional: resize to rgb 
                #img = cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
                #small_image = cv2.resize(img, dsize=(width, height), interpolation=cv2.INTER_CUBIC)
                print('fail')
            '''

            new_im.save(working_folder + '/' + label + '/' + str(i) + '.jpg')
            #cv2.imwrite(working_folder + '/' + label + '/' + str(i) + '.jpg', small_image)
    


In [ ]:
new_size = 0
for i in classes3:
    path_folder = '../working/adapted_images/' + i + '/'
    #get number of filed in the directory
    number_of_input_files = len(os.listdir(path_folder))
    print(number_of_input_files)
    new_size += number_of_input_files
print(new_size)


In [ ]:
'''check whether the saving of the resized images has worked -> show first picture in the folder LeonardodaVinci'''

path = '../working/adapted_images/LeonardodaVinci/'

img_path = os.listdir(path)[0]
print(img_path)
img = io.imread(path + img_path)

plt.figure()
plt.imshow(img)

In [ ]:
#define of dataset, trainset and testset 
'''source: https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html'''
path = '../working/adapted_images/'

transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.RandomHorizontalFlip(0.5), #reduces the risk of overfitting
     transforms.RandomVerticalFlip(0.5), #reduces the risk of overfitting
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])


dataset = torchvision.datasets.ImageFolder(root=path,transform=transform)


In [ ]:
#define trainset and testset 
'''source: https://www.kaggle.com/androbomb/using-cnn-to-classify-images-w-pytorch'''

dataset_size = new_size

#number of images in the dataset
#dataset_size = len(labels)

train_size = int(0.7 * dataset_size) #golden rule 70/30 for train and test set 
test_size = dataset_size - train_size

batch_size = 30
                 
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = torch.utils.data.DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
    
test_loader = torch.utils.data.DataLoader(test_dataset,batch_size=batch_size,shuffle=False)

In [ ]:
number_of_labels = len(classes3) #number of different labels -> defines output layer of the CNN
print(number_of_labels)

In [ ]:
# Create a neural net class
class Net(nn.Module):
    
    # Defining the Constructor
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=12, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=12, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.fc = nn.Linear(in_features=2000000, out_features=number_of_labels)

    def forward(self, x):
        x = F.relu(self.pool(self.conv1(x))) 
        x = F.relu(self.pool(self.conv2(x)))  
        #x = x.view(-1, 32 * 32 * 64)
        x = x.view(-1, 2000000)
        x = self.fc(x)
        x = torch.log_softmax(x, dim=1)
        return x
    
net = Net()

print(net)

In [ ]:
#Training 

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.0002) #performed better than SGD

epoch_count = 10
losses = []
accuracies = []

correct = 0
batch_total_count = 0

for epoch in range(epoch_count):
    train_loss = 0.0
    for i, (input_data, label_data) in enumerate(train_loader):        
        optimizer.zero_grad() # reset of opimizer
        output_data = net(input_data)
        loss = criterion(output_data, label_data) #forward
        loss.backward() #backward
        optimizer.step() #optimize
        loss = loss.item()
        train_loss += loss #sum up the loss of the epoch
        
        _, predicted = torch.max(output_data.data, 1)
        correct = torch.sum(label_data==predicted).item()
        acc = correct/len(label_data)
        losses.append(loss)
        accuracies.append(acc)
        print("batch",i,"\tloss: ", round(loss,3), "\taccuracy: ", round(acc*100,3),"%")
        
    loss_epoch = train_loss / train_size * batch_size
    print('loss in epoch', str(epoch +1) , ': ', loss_epoch)
 

In [ ]:
# plots the losses and accuracies from every batch from the training

fig, axs = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
axs[0].plot(range(len(losses)), losses,'-x')
axs[0].set_ylabel("Cross Entropy Loss", fontsize = 15)
axs[1].plot(range(len(accuracies)), [100*x for x in accuracies],'-rx')
axs[1].set_ylabel("Accuracy in Percent", fontsize = 15)
axs[1].set_xlabel("batch number",fontsize = 20)
fig.suptitle('Loss and Accuraccy metrics for training set',fontsize = 20)

plt.show()

In [ ]:
#Evaluation

net.eval()
correct = 0
total = 0

truelabels = []
predictions = []

with torch.no_grad():
    for data, target in test_loader:
        input_data, label_data = Variable(data), Variable(target)
        output_data = net(input_data)
        _, predicted = torch.max(output_data.data, 1)
        correct += torch.sum(target==predicted).item()
        total += label_data.size(0)

        for label in target.data.numpy():
            truelabels.append(label)
        for prediction in net(data).data.numpy().argmax(1):
            predictions.append(prediction) 

    accuracy = correct / test_size
    print('accuracy in %: ', str(accuracy*100))

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd
import seaborn as sns
import numpy as np

cm = confusion_matrix(truelabels, predictions)
tick_marks = np.arange(len(classes3))

df_cm = pd.DataFrame(cm, index = classes3, columns = classes3)
plt.figure(figsize = (15,15))
sns.heatmap(df_cm, annot=False, cmap=plt.cm.Blues, fmt='g')
plt.xlabel("Predicted Shape", fontsize = 20)
plt.ylabel("True Shape", fontsize = 20)
plt.show()

In [ ]:
import math
for data, target in test_loader:

        output_data = net(data)
        _, predicted = torch.max(output_data.data, 1)
        images = []
        
        l = len(input_data)
        fig, axs = plt.subplots(math.ceil(l/3), 3, figsize=(15, math.ceil(l/3)*5))
        fig.suptitle('Example predictions of testset',fontsize = 20)
        for i in range(l):
            img = input_data[i]
            img = img.permute(1, 2, 0)
            
            axs[i//3,i%3].imshow(img)
            axs[i//3,i%3].axis("off")
            axs[i//3,i%3].title.set_text(f'Prediction: {classes3[predicted[i]]} \n Actual target: {classes3[target[i]]}')
            
        plt.show()
           
        break


In [ ]:
#model saving
path = '../working/final_model'
torch.save(net.state_dict(), path)

In [ ]:
#model loading
#net2 = Net()
#net2.load_state_dict(torch.load("/kaggle/input/model1/modell"), strict=False)